In [ ]:
from huggingface_hub import login

login(token="")

In [ ]:
# ======================================================
# 1. Installer les libs
# ======================================================
!pip install transformers datasets peft accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 13.4 MB/s eta 0:00:00


In [ ]:
# ======================================================
# 1. Imports
# ======================================================
import torch
import pandas as pd
import json
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

# ======================================================
# 2. Configs
# ======================================================
MODEL_NAME = "google/gemma-2-2b-it"
DATA_FILE = "dataEN.jsonl"
OUTPUT_DIR = "./gemma-2-2b-it-usecase"
MAX_LENGTH = 1300
EPOCHS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 8

# ======================================================
# 3. Charger modèle et tokenizer
# ======================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# ======================================================
# 4. Charger dataset
# ======================================================
df = pd.read_json(DATA_FILE, lines=True)
dataset = Dataset.from_pandas(df)

# ======================================================
# 5. Formatter chaque ligne pour Use Case Diagram
# ======================================================
def format_usecase_data_jsonl(ex):
    # Rectangles / systèmes
    output_rectangles = [
        {
            "text": r.get("text", r.get("name", "N/A")),
            "name": r.get("name", "N/A")
        }
        for r in ex.get("output", {}).get("rectangle", [])
    ]

    # Acteurs
    output_actors = [
        {
            "text": a.get("text", a.get("name", "N/A")),
            "name": a.get("name", "N/A"),
            "alias": a.get("alias", "<no alias>")
        }
        for a in ex.get("output", {}).get("Actors", [])
    ]

    # Cas d’usage
    output_use_cases = [
        {
            "text": uc.get("text", uc.get("name", "N/A")),
            "name": uc.get("name", "N/A"),
            "alias": uc.get("alias", "N/A")
        }
        for uc in ex.get("output", {}).get("Use cases", [])
    ]

    # Relations
    output_relations = [
        {
            "from": r.get("from", "N/A"),
            "to": r.get("to", "N/A"),
            "type": r.get("type", "N/A")
        }
        for r in ex.get("output", {}).get("Relations", [])
    ]

    # Construire le JSON final
    json_line = {
        "instruction": ex.get("instruction", ""),
        "input": ex.get("input", ""),
        "output": {
            "rectangles": output_rectangles,
            "actors": output_actors,
            "use_cases": output_use_cases,
            "relations": output_relations
        }
    }

    return {"text": json.dumps(json_line, ensure_ascii=False)}

formatted_dataset = dataset.map(format_usecase_data_jsonl)

# ======================================================
# 6. Tokenization
# ======================================================
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names
)

# ======================================================
# 7. Préparer LoRA
# ======================================================
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=128,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # or SEQ_2_SEQ_LM
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# ======================================================
# 8. TrainingArguments
# ======================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
)

# ======================================================
# 9. Trainer avec DataCollator
# ======================================================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator,
)

# ======================================================
# 10. Lancer fine-tuning
# ======================================================
trainer.train()

# ======================================================
# 11. Sauvegarde
# ======================================================
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Modèle et tokenizer sauvegardés dans {OUTPUT_DIR}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Map:   0%|          | 0/602 [00:00<?, ? examples/s]

trainable params: 51,118,080 || all params: 2,665,459,968 || trainable%: 1.9178


It is strongly recommended to train Gemma2 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.111300
20,0.863100
30,0.840400
40,0.795000
50,0.755200
60,0.679600
70,0.726600
80,0.607100
90,0.712800
100,0.620700


Modèle et tokenizer sauvegardés dans ./gemma-2-2b-it-usecase


In [ ]:
from google.colab import drive
import os

# 1️⃣ Monter Google Drive
drive.mount('/content/drive')

# 2️⃣ Créer le dossier Modeles s'il n'existe pas
drive_base_path = "/content/drive/MyDrive/Modeles"
os.makedirs(drive_base_path, exist_ok=True)

# 3️⃣ Chemin complet pour ton modèle
drive_model_path = os.path.join(drive_base_path, "gemma-2-2b-it-usecase")
os.makedirs(drive_model_path, exist_ok=True)

# 4️⃣ Sauvegarder le modèle et tokenizer
peft_model.save_pretrained(drive_model_path)
tokenizer.save_pretrained(drive_model_path)

print(f"Modèle et tokenizer sauvegardés dans : {drive_model_path}")


Mounted at /content/drive
Modèle et tokenizer sauvegardés dans : /content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase


In [ ]:
from google.colab import drive
import os

# 1️⃣ Monter Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ======================================================
# 1. Imports
# ======================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time  # <-- pour mesurer le temps

# ======================================================
# 2. Config
# ======================================================
MODEL_DIR = "/content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase-fine-tuned"  # chemin vers ton modèle
MAX_LENGTH = 1600

# ======================================================
# 3. Charger le modèle et tokenizer sur GPU
# ======================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,  # float16 sur GPU
    device_map="auto",  # répartit sur GPU automatiquement
    trust_remote_code=True,
    local_files_only=True
)
model.to(device)
model.eval()

# ======================================================
# 4. Fonction d'inférence avec mesure du temps
# ======================================================
def predict(input_text):
    prompt = (
        f"Convert the following text into a structured JSON representing the UML use case diagram, "
        f"including rectangles , actors, use cases, and relations.\nInput: {input_text}\nOutput:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

    start_time = time.time()  # début du chronomètre

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_LENGTH,
            temperature=0.0,  # déterministe
            do_sample=False
        )

    end_time = time.time()  # fin du chronomètre
    generation_time = end_time - start_time  # calcul en secondes

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"\n⏱ Temps de génération : {generation_time:.2f} secondes")  # affichage du temps

    return decoded.split("Output:")[-1].strip() if "Output:" in decoded else decoded.strip()

# ======================================================
# 5. Tester avec input
# ======================================================
while True:
    user_input = input("\nEntrez votre texte (ou 'exit' pour quitter) : ")
    if user_input.lower() == "exit":
        break

    prediction = predict(user_input)
    print("\n=== Predicted Structured Output ===")
    print(prediction)


Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Entrez votre texte (ou 'exit' pour quitter) : The Textile Management Systemcase study is designed to monitor all the processes in a textile manufacturing industry. There are various processes such as order receiving, purchase of raw materials, classification of raw materials, production of goods and selling the products. The main aim of this system is to ensure the reliability of data and to ease the process of manufacturing and transportation of garments and raw materials. This system ensures that the hardships prevailing in the manual systems are overridden by reducing the errors in data handling. The major functionalities are customer registration, sampling, stock checking, purchasing and processing of raw materials, production of garments, product checking and clearance, shipment, billing and report generation.

⏱ Temps de génération : 84.74 secondes

=== Predicted Structured Output ===
The Textile Management System use case diagram.

Actors,Customer,Purchasing Manager,Production 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import os

# --- CONFIGURATION ---
base_model = "google/gemma-2-2b-it"
lora_path = "/content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase"
save_dir = "/content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase-fine-tuned"

use_cuda = torch.cuda.is_available()
dtype = torch.float16 if use_cuda else torch.float32
device_map = "auto" if use_cuda else {"": "cpu"}

# --- Vérifier dossier LoRA ---
print("Vérification du dossier LoRA :", lora_path)
if not os.listdir(lora_path):
    raise FileNotFoundError(f"Le dossier LoRA {lora_path} est vide ! Vérifie le montage de Drive.")

assert os.path.isfile(os.path.join(lora_path, "adapter_config.json")), \
    "adapter_config.json introuvable dans le dossier LoRA !"

print("Contenu du dossier LoRA :", os.listdir(lora_path))

# --- Chargement du tokenizer ---
print("Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)

# --- Chargement du modèle de base ---
print("Chargement du modèle de base...")
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=dtype,
    device_map=device_map,
    low_cpu_mem_usage=True
)

# --- Chargement du LoRA ---
print(f"Chargement du LoRA depuis : {lora_path}")
peft_model = PeftModel.from_pretrained(
    model,
    lora_path,
    device_map=device_map,
    local_files_only=True,
    repo_type="local_folder"  # Clé pour forcer lecture locale
)

# --- Fusion LoRA dans le modèle de base ---
print("Fusion des poids LoRA dans le modèle de base...")
peft_model.merge_and_unload()
merged_model = model

# --- Sauvegarde du modèle fusionné ---
print(f"Sauvegarde du modèle fusionné dans : {save_dir}")
merged_model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

print("✅ Fusion terminée !")
print(f"Modèle fusionné sauvegardé dans : {save_dir}")


Vérification du dossier LoRA : /content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase
Contenu du dossier LoRA : ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.model', 'tokenizer.json']
Chargement du tokenizer...
Chargement du modèle de base...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Chargement du LoRA depuis : /content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase
Fusion des poids LoRA dans le modèle de base...
Sauvegarde du modèle fusionné dans : /content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase-fine-tuned
✅ Fusion terminée !
Modèle fusionné sauvegardé dans : /content/drive/MyDrive/Modeles/gemma-2-2b-it-usecase-fine-tuned
